In [18]:
import pandas as pd
import numpy as np

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

Pandas version: 3.0.6
NumPy version: 2.5.3


In [19]:
df = pd.read_csv("../data/raw/HHS_Unaccompanied_Alien_Children_Program.csv")
df.head()

,Date,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care
0,"December 21, 2025",6.0,18.0,11.0,"2,484",14.0
1,"December 18, 2025",11.0,50.0,6.0,"2,472",16.0
2,"December 17, 2025",7.0,31.0,11.0,"2,481",10.0
3,"December 16, 2025",8.0,54.0,15.0,"2,468",9.0
4,"December 15, 2025",11.0,42.0,9.0,"2,470",7.0


In [20]:
df.shape

(1170, 6)

In [21]:
df.columns

Index(['Date', 'Children apprehended and placed in CBP custody*',
       'Children in CBP custody', 'Children transferred out of CBP custody',
       'Children in HHS Care', 'Children discharged from HHS Care'],
      dtype='str')

In [22]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1170 entries, 0 to 1169
Data columns (total 6 columns):
 #   Column                                           Non-Null Count  Dtype  
---  ------                                           --------------  -----  
 0   Date                                             720 non-null    str    
 1   Children apprehended and placed in CBP custody*  720 non-null    float64
 2   Children in CBP custody                          720 non-null    float64
 3   Children transferred out of CBP custody          720 non-null    float64
 4   Children in HHS Care                             720 non-null    str    
 5   Children discharged from HHS Care                720 non-null    float64
dtypes: float64(4), str(2)
memory usage: 69.5 KB


In [23]:
df.describe(include="all")

,Date,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care
count,720,720.000000,720.000000,720.000000,720,720.000000
unique,720,NaN,NaN,NaN,642,NaN
top,"December 21, 2025",NaN,NaN,NaN,"2,340",NaN
freq,1,NaN,NaN,NaN,3,NaN
mean,NaN,93.523611,171.494444,128.668056,NaN,173.406944
std,NaN,72.646625,126.354965,97.322012,NaN,125.702841
min,NaN,0.000000,7.000000,0.000000,NaN,0.000000
25%,NaN,12.000000,36.000000,14.000000,NaN,19.750000
50%,NaN,99.000000,193.000000,157.000000,NaN,181.000000
75%,NaN,147.250000,263.250000,199.250000,NaN,267.000000


In [24]:
df.columns.tolist()

['Date',
 'Children apprehended and placed in CBP custody*',
 'Children in CBP custody',
 'Children transferred out of CBP custody',
 'Children in HHS Care',
 'Children discharged from HHS Care']

In [25]:
df.isnull().sum()

Date                                               450
Children apprehended and placed in CBP custody*    450
Children in CBP custody                            450
Children transferred out of CBP custody            450
Children in HHS Care                               450
Children discharged from HHS Care                  450
dtype: int64

In [26]:
df = df.dropna(how="all").copy()

In [27]:
df.shape

(720, 6)

In [28]:
df.columns = df.columns.str.strip()

In [29]:
df["Date"] = pd.to_datetime(
    df["Date"],
    errors="coerce"
)

In [30]:
df["Date"].min()

Timestamp('2023-01-12 00:00:00')

In [31]:
df["Date"].max()

Timestamp('2025-12-21 00:00:00')

In [32]:
df = df.sort_values("Date").reset_index(drop=True)

In [33]:
df.head()

,Date,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care
0,2023-01-12,33.0,53.0,34.0,"6,566",436.0
1,2023-01-22,32.0,49.0,39.0,"7,122",227.0
2,2023-01-23,32.0,50.0,39.0,"7,280",181.0
3,2023-01-24,47.0,42.0,47.0,"7,433",175.0
4,2023-01-25,20.0,22.0,41.0,"7,538",180.0


In [34]:
numeric_columns = [
    "Children apprehended and placed in CBP custody*",
    "Children in CBP custody",
    "Children transferred out of CBP custody",
    "Children in HHS Care",
    "Children discharged from HHS Care"
]

for col in numeric_columns:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", "", regex=False)
    )

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

In [35]:
df.dtypes

Date                                               datetime64[us]
Children apprehended and placed in CBP custody*           float64
Children in CBP custody                                   float64
Children transferred out of CBP custody                   float64
Children in HHS Care                                        int64
Children discharged from HHS Care                         float64
dtype: object

In [36]:
df.isnull().sum()

Date                                               0
Children apprehended and placed in CBP custody*    0
Children in CBP custody                            0
Children transferred out of CBP custody            0
Children in HHS Care                               0
Children discharged from HHS Care                  0
dtype: int64

In [37]:
df["Date"].duplicated().sum()

np.int64(0)

In [38]:
full_dates = pd.date_range(
    start=df["Date"].min(),
    end=df["Date"].max(),
    freq="D"
)

missing_dates = full_dates.difference(df["Date"])

print("Expected calendar dates:", len(full_dates))
print("Observed dates:", df["Date"].nunique())
print("Missing dates:", len(missing_dates))

Expected calendar dates: 1075
Observed dates: 720
Missing dates: 355


In [42]:
full_dates = pd.date_range(
    start=df["Date"].min(),
    end=df["Date"].max(),
    freq="D"
)

missing_dates = full_dates.difference(df["Date"])

print("Expected calendar dates:", len(full_dates))
print("Observed dates:", df["Date"].nunique())
print("Missing dates:", len(missing_dates))

Expected calendar dates: 1075
Observed dates: 720
Missing dates: 355


In [43]:
df["transfer_invalid"] = (
    df["Children transferred out of CBP custody"]
    > df["Children in CBP custody"]
)

In [44]:
df["transfer_invalid"].sum()

np.int64(86)

In [45]:
df[df["transfer_invalid"]]

,Date,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care,transfer_invalid
3,2023-01-24,47.0,42.0,47.0,7433,175.0,True
4,2023-01-25,20.0,22.0,41.0,7538,180.0,True
9,2023-02-02,15.0,13.0,23.0,7879,298.0,True
22,2023-02-22,107.0,215.0,230.0,7978,232.0,True
23,2023-02-23,101.0,162.0,178.0,7914,386.0,True
...,...,...,...,...,...,...,...
502,2025-01-30,47.0,42.0,47.0,3923,159.0,True
503,2025-02-02,20.0,22.0,41.0,3483,168.0,True
508,2025-02-09,15.0,13.0,23.0,2878,99.0,True
512,2025-02-13,15.0,10.0,23.0,2703,72.0,True


In [46]:
df["discharge_invalid"] = (
    df["Children discharged from HHS Care"]
    > df["Children in HHS Care"]
)

In [47]:
df["discharge_invalid"].sum()

np.int64(0)

In [48]:
df["data_quality_flag"] = (
    df["transfer_invalid"] |
    df["discharge_invalid"]
)

In [49]:
df["data_quality_flag"].sum()

np.int64(86)

In [50]:
for col in numeric_columns:
    print(
        col,
        "negative values:",
        (df[col] < 0).sum()
    )

Children apprehended and placed in CBP custody* negative values: 0
Children in CBP custody negative values: 0
Children transferred out of CBP custody negative values: 0
Children in HHS Care negative values: 0
Children discharged from HHS Care negative values: 0


In [51]:
df.to_csv(
    "../data/processed/uac_cleaned.csv",
    index=False
)